In [ ]:
!pip install -q opencv-contrib-python

In [ ]:
!pip install -q ai-accelerator

In [ ]:
!pip install -q torchinfo

In [ ]:
!pip show torchinfo

In [ ]:
!pip show opencv-contrib-python

In [ ]:
!pip show ai-accelerator

In [ ]:
import tarfile
import numpy
from pathlib import Path
import os
import matplotlib.pyplot as plt
import pickle
import random
import cv2 as cv
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as op
from torch.utils.data import DataLoader
from google.colab import files

In [ ]:
folder = Path()
file_name = "output_dir"
path_imgs = folder / file_name
path_imgs.mkdir(parents=True,exist_ok=True,mode=700)

file_to_download = 'cifar-100-python.tar.gz'
download_url = 'https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz'

if not os.path.exists(os.path.join('/content/', file_to_download)):
    print(f'Downloading {file_to_download}...')
    !wget {download_url} -P /content/
    if os.path.exists(os.path.join('/content/', file_to_download)):
        print(f'Successfully downloaded {file_to_download}.')
    else:
        print(f'Download of {file_to_download} failed.')
else:
    print(f'{file_to_download} already exists in /content/. Skipping download.')

try :
  with tarfile.open("/content/cifar-100-python.tar.gz","r:gz") as tar:
     tar.extractall(path=path_imgs)

     for member in tar.getmembers():
        # Construct the full path to the extracted member on the filesystem
        extracted_member_path = path_imgs / member.name

        if member.isdir():
           # Check if it's an actual directory on the filesystem after extraction
           if extracted_member_path.is_dir():
               for f_name in os.listdir(extracted_member_path):
                   print(f"name of file in directory {extracted_member_path}: {f_name}")
                   file_to_open_path = extracted_member_path / f_name
                   if file_to_open_path.is_file():
                       try:
                           with open(file_to_open_path,"rb") as file_handle:
                               print(f"Content of {file_to_open_path}:")
                               data_dict = pickle.load(file_handle,encoding="latin1")
                       except UnicodeDecodeError:
                           print(f"Skipping {file_to_open_path}: Cannot decode as text (likely binary file).")
                       except Exception as e:
                           print(f"Error reading {file_to_open_path}: {e}")
except:
  print("Something went wrong ! maybe you have load /content/cifar-100-python.tar.gz on google colab.")


In [ ]:
data_dict.keys()

In [ ]:
print("INFO:\n")
print("A list of the actual filenames for the images: ")
print(f"FILENAMES : {set(data_dict["filenames"])}")
print(f"NUMBER OF IMAGES : {len(set(data_dict["filenames"]))}")
print("\n")
print("A list of 100 specific class indices (0–99): ")
print(f"FINE_LABEL : {set(data_dict["fine_labels"])}")
print("\n")
print("A list of 20 superclass indices (0–19): ")
print(f"COARSE_LABELS : {set(data_dict["coarse_labels"])}")
print("\n")
print(f"DATA SHAPE : {data_dict["data"].shape}")
print(f" 3072 is the shape of a single CIFAR-100 image when it is \"flattened\" into a 1D vector.")
print("\n")
print(f"""This number comes from the product of the image's dimensions:
         Width: 32 pixels
         Height: 32 pixels
         Channels: 3 (Red, Green, Blue)\n
         Total Calculation: 32*32*3 = {32*32*3} """)



In [ ]:

  # Reshape the 1D data (3072,) into (3, 32, 32) for channels, height, width
  # Then transpose it to (32, 32, 3) which is the expected format for matplotlib imshow
  # Pick 3 random indices
print("""The images are pixelated because the CIFAR-100 dataset is made up of very low-resolution images (32x32 pixels).
When these small images are displayed on a larger screen, each pixel of the original image is stretched,
which gives a pixelated look.This is an intentional feature of this dataset.\n""")

indices = random.sample(range(len(data_dict["data"])), k=3)
plt.figure(figsize=(9, 3))
for i, img_index in enumerate(indices):
    img = data_dict["data"][img_index].reshape(3, 32, 32).transpose(1, 2, 0)
    plt.subplot(1, 3, i+1)
    plt.imshow(img,interpolation="bicubic")
    plt.axis("off")

plt.tight_layout()
plt.show()



In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(degrees=30),
    transforms.RandomAffine(degrees=0,translate=(0.1,0.1),scale=(0.8,1.5)),
    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2),

])

In [ ]:
print("""SIFT (Scale-Invariant Feature Transform) is a computer vision algorithm
used to detect and describe distinctive features in images that remain
stable under changes in scale, rotation, and illumination. """)

In [ ]:

ScaleInvariantFeatureTransform = {}
sift = cv.SIFT_create()

for img , path in zip(data_dict["data"],data_dict["filenames"]):
  # Reshape 1D data to HWC format for color image
  img_reshaped_transposed = img.reshape(3, 32, 32).transpose(1, 2, 0) # Shape: (32, 32, 3)

  # Convert to grayscale. cv.cvtColor expects HWC for color input.
  gray_img = cv.cvtColor(img_reshaped_transposed, cv.COLOR_BGR2GRAY) # Shape: (32, 32) uint8

  # Apply torchvision transforms.
  img_transformed_tensor = transform(gray_img)

  # Convert the PyTorch tensor back to a NumPy array for OpenCV's SIFT.
  # Squeeze the channel dimension and convert to uint8 (scaling from [0,1] to [0,255]).
  # SIFT typically works best with uint8 images.
  img_for_sift = (img_transformed_tensor.squeeze(0).numpy() * 255).astype(numpy.uint8) # (32, 32) uint8

  # Detect and compute SIFT features on the processed NumPy image
  keypoints , descriptor = sift.detectAndCompute(img_for_sift, None)

  ScaleInvariantFeatureTransform[path] = [keypoints,descriptor]
  if keypoints is not None and  descriptor is not None:
    print(f"Number of keypoints for image {path} : {len(keypoints)}")
    print(f"Descriptor shape for image {path} : {descriptor.shape}")
  else :
    print(f"No keypoints or descriptor found for image {path}")

In [ ]:
filename_to_display = random.choice(data_dict["filenames"])
keypoint, descriptor = ScaleInvariantFeatureTransform[filename_to_display]

# Find the index of the filename in data_dict["filenames"]
img_index = data_dict["filenames"].index(filename_to_display)

# Get the raw image data from data_dict
raw_img_data = data_dict["data"][img_index]

# Reshape the 1D data to (3, 32, 32) and then transpose to (32, 32, 3) for OpenCV/Matplotlib
img_to_draw = raw_img_data.reshape(3, 32, 32).transpose(1, 2, 0)

# Ensure the image data is uint8, which is expected by cv.drawKeypoints
img_to_draw = img_to_draw.astype(numpy.uint8)

# Draw keypoints on the reconstructed image
img_with_keypoints = cv.drawKeypoints(img_to_draw, keypoint, None, flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

# Display the image with keypoints using matplotlib instead of cv.imshow
plt.figure(figsize=(4, 4))
plt.imshow(img_with_keypoints,interpolation="bicubic")
plt.title(f"Keypoints for {filename_to_display}")
plt.axis('off')
plt.show()

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
  """
    batch: list of samples from DataLoader. Each sample is a tuple: (filename, [keypoints, descriptor])
    return: (list of filenames, padded_batch of descriptors with shape (batch_size, max_num_keypoints, descriptor_dim))
  """
  filenames = [] # Store filenames
  descriptors_to_pad = []
  for filename , kp_desc_pair in batch: # kp_desc_pair is [keypoints_tuple, descriptor_ndarray]
    descriptor_ndarray = kp_desc_pair[1] # This extracts the actual numpy array (or None)
    filenames.append(filename) # Collect filename

    if descriptor_ndarray is not None:
      # Convert numpy array to torch tensor, ensure float32 type as SIFT descriptors are floats
      descriptors_to_pad.append(torch.from_numpy(descriptor_ndarray).float())
    else:
      # If an image has no descriptors (descriptor_ndarray is None), append an empty tensor
      # This ensures pad_sequence can still process the batch, and we don't drop samples.
      # The empty tensor should have the correct descriptor dimension (128 for SIFT).
      descriptors_to_pad.append(torch.empty(0, 128))

  # Pad the sequence of tensors to the max length in the batch
  padded_batch = pad_sequence(descriptors_to_pad, batch_first=True, padding_value=0.0)
  return filenames, padded_batch # Return both filenames and padded batch

dataset = DataLoader(list(ScaleInvariantFeatureTransform.items()), batch_size=32, collate_fn=collate_fn)

# To get a batch from the DataLoader, you need to iterate over it
# Get the first batch to inspect its shape
first_batch_filenames, first_batch_descriptors = next(iter(dataset))
print(f"first batch filenames (first 5): {first_batch_filenames[:5]}")
print(f"first batch descriptors shape: {first_batch_descriptors.shape}")

In [ ]:
class EmbedInvariantPrintDescriptor(nn.Module):
  def __init__(self):
    super().__init__()
    # Initialize linear layers for Q, K, V projections instead of fixed random tensors
    self.q_proj = nn.Linear(128, 128)
    self.k_proj = nn.Linear(128, 128)
    self.v_proj = nn.Linear(128, 128)

    self.attention = nn.MultiheadAttention(embed_dim=128, num_heads=8, batch_first=False) # MultiheadAttention expects (seq_len, batch_size, embed_dim) by default
    self.noisestd = 0.1

    # Final feed-forward layers for processing the aggregated descriptor
    self.final_linear1 = nn.Linear(128, 256)
    self.relu = nn.ReLU()
    self.final_linear2 = nn.Linear(256, 128)
    self.batch_norm = nn.BatchNorm1d(128) # BatchNorm for (batch_size, num_features)

  def forward(self, x_batch): # x_batch has shape (batch_size, max_num_keypoints, 128)
    output_vectors = []
    for x in x_batch: # x for a single image has shape (num_keypoints, 128)

      if x.shape[0] == 0: # Handle images with no keypoints
          # Append a zero vector of the expected final dimension (256)
          output_vectors.append(torch.zeros(128, device=x.device)) # Changed output size to 128 here
          continue

      # Add noise for perturbation
      x_perturbed = x + torch.rand_like(x) * self.noisestd

      # Project x_perturbed into Query, Key, Value
      # Each projection will have shape (num_keypoints, 128)
      query = self.q_proj(x_perturbed)
      key = self.k_proj(x_perturbed)
      value = self.v_proj(x_perturbed)

      # MultiheadAttention expects inputs of shape (seq_len, batch_size, embed_dim)
      # For a single image, seq_len is num_keypoints, and batch_size is 1
      query = query.unsqueeze(1) # Shape: (num_keypoints, 1, 128)
      key = key.unsqueeze(1)    # Shape: (num_keypoints, 1, 128)
      value = value.unsqueeze(1)  # Shape: (num_keypoints, 1, 128)

      # Perform multihead attention
      # attention_output will have shape (num_keypoints, 1, 128)
      attention_output, _ = self.attention(query, key, value)

      # Aggregate attention output to get a single vector per image
      # Squeeze the batch_size dimension: (num_keypoints, 128)
      # Then take the mean along the num_keypoints dimension: (128,)
      aggregation = attention_output.squeeze(1).mean(dim=0) # Resulting shape: (128,)

      # Pass through the final feed-forward network
      output1 = self.final_linear1(aggregation) # Shape: (256,)
      activation_fn = self.relu(output1)
      output2 = self.final_linear2(activation_fn) # Shape: (128,)
      output_vectors.append(output2)

    # Stack all individual image vectors into a batch tensor
    final_batch_tensor = torch.stack(output_vectors) # Shape: (batch_size, 128)

    # Apply batch normalization to the final batch tensor
    return self.batch_norm(final_batch_tensor)

In [ ]:
from tqdm.auto import tqdm

torch.manual_seed(42)
epochs = 50

# Instantiate the model once outside the training loop
model = EmbedInvariantPrintDescriptor()
optimizer = op.Adam(model.parameters(), lr=0.001) # Define an optimizer
loss_fn = nn.L1Loss() # Define a loss function

for epoch in tqdm(range(epochs)):
  model.train() # Set model to training mode
  total_loss = 0
  for batch in dataset:
    optimizer.zero_grad() # Zero the gradients

    vect = model(batch) # Pass the batch to the model's forward method
    loss = loss_fn(vect, batch.mean(dim=1)) # Calculate loss

    loss.backward() # Backpropagate
    optimizer.step() # Update weights
    total_loss += loss.item()

  loss_per_epoch = total_loss / len(dataset)
  if epoch % 5 == 0:
     print(f"Epoch {epoch+1}/{epochs}, Loss: {loss_per_epoch:.2f}%")
     print(f"------------------------------------------------")

In [ ]:
from tqdm.auto import tqdm

# Assuming `model` is already trained and available
model.eval() # Set the model to evaluation mode

all_embeddings = []
all_filenames = []

with torch.inference_mode(): # Disable gradient calculations
    for filenames, descriptors_batch in tqdm(dataset, desc="Generating Embeddings"):
        embeddings = model(descriptors_batch)
        all_embeddings.append(embeddings.cpu().numpy()) # Move to CPU and convert to NumPy
        all_filenames.extend(filenames)

# Concatenate all embeddings into a single NumPy array
all_embeddings_np = numpy.vstack(all_embeddings)

print(f"Generated {all_embeddings_np.shape[0]} embeddings, each with dimension {all_embeddings_np.shape[1]}.")
print(f"Total filenames collected: {len(all_filenames)}")


In [ ]:
import torchinfo

summary = torchinfo.summary(model)
print(summary)

### Upload your image

Use the file uploader below to provide your image. Once uploaded, the code will generate its invariant embedding.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

# Get the filename of the uploaded image
if uploaded:
    for fn in uploaded.keys():
        uploaded_image_path = os.path.join('/content', fn)
        print(f'User uploaded file "{fn}" at "{uploaded_image_path}"')
        break # Assuming only one file will be processed
else:
    print("No file uploaded.")

In [ ]:
def generate_embedding_using_EmbedInvariantPrintDescriptor(image_path, model):
    import cv2 as cv
    import numpy
    import torch

    img = cv.imread(image_path)

    if img is None:
        return f"Error: Could not load image from {image_path}"

    # Ensure image is in 3 channels for consistent grayscale conversion
    if len(img.shape) == 2: # Grayscale image
        img_rgb = cv.cvtColor(img, cv.COLOR_GRAY2BGR)
    elif img.shape[2] == 4: # RGBA image
        img_rgb = cv.cvtColor(img, cv.COLOR_RGBA2BGR)
    else: # BGR image
        img_rgb = img

    sift = cv.SIFT_create()
    img_gray = cv.cvtColor(img_rgb, cv.COLOR_BGR2GRAY)
    keypoints, descriptor = sift.detectAndCompute(img_gray, None)

    if descriptor is not None and len(descriptor) > 0:
        descriptor_tensor = torch.from_numpy(descriptor).float()
        with torch.inference_mode():
            model.eval()
            return model(descriptor_tensor.unsqueeze(0))
    else:
        # Return a zero embedding if no keypoints are found
        with torch.inference_mode():
            model.eval()
            device = next(model.parameters()).device if next(model.parameters(), None) is not None else 'cpu'
            return torch.zeros(1, 128, device=device)


In [ ]:
if 'uploaded_image_path' in locals() and uploaded_image_path:
    user_image_embedding = generate_embedding_using_EmbedInvariantPrintDescriptor(uploaded_image_path, model)
    print(f"Generated embedding for uploaded image: {user_image_embedding}")
    print(f"Embedding shape: {user_image_embedding.shape}")
else:
    print("No image was uploaded to generate an embedding.")

In [ ]:
torch.save(model.state_dict(), "model_EmbedInvariantPrintDescriptor.pth")